# اختبار MADCrowner على Google Colab GPU (عينة واحدة)

> **تنبيه خصوصية مهم:** تشغيل هذا الدفتر يرفع ملفات الحالة من Google Drive إلى جلسة Google Colab السحابية لمعالجتها. استخدمه فقط إذا كان مسموحًا لك بنقل هذه العينة إلى Google. لا يحتوي المستودع على ملفات `crown.ply` و`pna_crop.ply`؛ لا ترفع meshes سريرية إلى GitHub.

هذا فحص تقني فقط، وليس تدريبًا صالحًا للاستخدام السريري. لا توجد أوزان MADCrowner مدرّبة في المستودع؛ يبدأ الاختبار بأوزان عشوائية ويحفظ checkpoint تجريبيًا لا يصلح لتصميم تاج. سيشغّل الدفتر `IOS_Dataset` مع PyTorch3D FPS الأصلي، ثم forward/backward وتحديث optimizer عشر مرات على العينة نفسها.

## قبل التشغيل
1. من Colab اختر **Runtime → Change runtime type → T4 GPU** (إذا كان متاحًا لحسابك). لا تحتاج تغيير Python في Colab؛ الدفتر ينشئ بيئة Conda مستقلة بـPython 3.11 ويشغّل اختبارات GPU منها.
2. أضف GitHub token بصلاحية **Contents: Read-only** إلى Colab Secrets باسم `GITHUB_TOKEN`، أو أدخله عند مطالبة الدفتر. لا تضع الرمز داخل خلية أو ترفعه إلى GitHub.
3. أنشئ في Google Drive مجلد `MyDrive/DentalAI/`، وضع داخله `case_46_candidate_01.zip` الذي يحتوي الملفات الخمسة: `crown.ply`, `pna_crop.ply`, `boundary_labels.npy`, `crown_attributes.h5`, و`psr_my.npz`. يمكن أن تكون الملفات في جذر ZIP أو داخل مجلد واحد.
4. شغّل الخلايا بالترتيب. يثبت الدفتر Miniconda في الجلسة، وينشئ `/usr/local/envs/py311`، ثم يثبت PyTorch 2.4.1/CUDA 12.1 ويبني PyTorch3D من المصدر داخل البيئة. ينفّذ FPS وDataLoader والتدريب عبر `/usr/local/envs/py311/bin/python`، وليس Python 3.13 الخاص بنواة Colab. تثبيت CUDA Toolkit وبناء PyTorch3D قد يستغرقان وقتًا ومساحة إضافيين. إذا تعذر البناء أو لم يتوفر T4، يتوقف الدفتر برسالة خطأ.

المراجع: [تعليمات تثبيت PyTorch3D](https://github.com/facebookresearch/pytorch3d/blob/main/INSTALL.md) و[دليل PyTorch لاستخدام Google Colab وT4](https://docs.pytorch.org/tutorials/beginner/colab.html).

In [ ]:
import os, sys, subprocess
print('Python:', sys.version.split()[0])
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip()
print('GPU:', gpu_info)
if 'T4' not in gpu_info.upper():
    raise RuntimeError('لم يتم تخصيص NVIDIA T4. اختر Runtime > Change runtime type > T4 GPU ثم أعد التشغيل.')

## 1) استنساخ المستودع الخاص
يستخدم هذا القسم header مؤقتًا في عملية `git clone` فقط؛ لا يكتب الرمز في عنوان remote أو ملفات المستودع. إذا استُخدم Colab Secret، فعّل خيار Notebook access لذلك السر.

In [ ]:
import base64, getpass, subprocess
from pathlib import Path
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = None
if not gh_token:
    gh_token = getpass.getpass('GitHub token (Contents: Read-only): ')
if not gh_token:
    raise RuntimeError('يلزم GitHub token لقراءة المستودع الخاص.')
REPO_DIR = Path('/content/dental-crown-ai')
if not REPO_DIR.exists():
    basic = base64.b64encode(('x-access-token:' + gh_token).encode()).decode()
    env = os.environ.copy()
    env['GIT_CONFIG_COUNT'] = '1'
    env['GIT_CONFIG_KEY_0'] = 'http.extraheader'
    env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + basic
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/eliasajeep1966-gif/dental-crown-ai.git', str(REPO_DIR)], env=env, check=True)
    del env, basic
gh_token = None
print('Repository ready:', REPO_DIR)

## 2) إنشاء Conda Python 3.11 وتثبيت PyTorch3D

نواة Colab الحالية قد تعمل بـPython 3.13 وTorch أحدث من المتوافق مع PyTorch3D. لذلك لا نغيّرها ولا نثبت الحزم فيها: ننشئ `/usr/local/envs/py311`، ونثبت Python 3.11 وCUDA Toolkit 12.1 وPyTorch 2.4.1+cu121، ثم نبني PyTorch3D من المصدر. جميع عمليات النموذج أدناه تُنفذ باستخدام مفسر هذه البيئة عبر `subprocess`.


In [ ]:
import os, shutil, subprocess, urllib.request
from pathlib import Path

MINICONDA_DIR = Path('/usr/local/miniconda')
CONDA_EXE = MINICONDA_DIR / 'bin' / 'conda'
CONDA_ENV = Path('/usr/local/envs/py311')
CONDA_PY = CONDA_ENV / 'bin' / 'python'
INSTALLER = Path('/content/Miniconda3-latest-Linux-x86_64.sh')

if not CONDA_EXE.is_file():
    if MINICONDA_DIR.exists():
        shutil.rmtree(MINICONDA_DIR)
    if not INSTALLER.is_file():
        print('Downloading Miniconda installer...')
        urllib.request.urlretrieve(
            'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
            INSTALLER,
        )
    subprocess.run(['bash', str(INSTALLER), '-b', '-p', str(MINICONDA_DIR)], check=True)

if not CONDA_PY.is_file():
    subprocess.run([
        str(CONDA_EXE), 'create', '--yes', '--prefix', str(CONDA_ENV),
        '--override-channels', '-c', 'conda-forge', 'python=3.11', 'pip',
    ], check=True)

# Compile against the matching CUDA 12.1 toolkit, not the host image's newer nvcc.
subprocess.run([
    str(CONDA_EXE), 'install', '--yes', '--prefix', str(CONDA_ENV),
    '--override-channels', '-c', 'nvidia', '-c', 'conda-forge', 'cuda-toolkit=12.1',
], check=True)
subprocess.run([
    str(CONDA_PY), '-c',
    'import sys; print("Isolated Python:", sys.version); assert sys.version_info[:2] == (3, 11)',
], check=True)

def pip_install(*args):
    subprocess.run([str(CONDA_PY), '-m', 'pip', 'install', '--no-cache-dir', *args], check=True)

pip_install('setuptools', 'wheel', 'ninja')
pip_install('--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.4.1', 'torchvision==0.19.1')
pip_install(
    'numpy==1.26.4', 'scipy==1.15.3', 'h5py==3.14.0', 'trimesh==4.8.1',
    'PyYAML==6.0.2', 'tqdm==4.67.1', 'imageio==2.37.0', 'plyfile==1.1.2',
    'scikit-image==0.25.2', 'open3d==0.19.0', 'libigl', 'ipdb', 'fvcore',
    'iopath', 'torch-geometric==2.6.1',
)
pip_install(
    '--find-links', 'https://data.pyg.org/whl/torch-2.4.1+cu121.html',
    'torch-scatter', 'torch-cluster',
)

build_env=os.environ.copy()
build_env['FORCE_CUDA']='1'
build_env['MAX_JOBS']='2'
build_env['TORCH_CUDA_ARCH_LIST']='7.5'  # NVIDIA T4
build_env['CUDA_HOME']=str(CONDA_ENV)
build_env['PATH']=str(CONDA_ENV/'bin')+os.pathsep+build_env.get('PATH','')
lib_dirs=[str(CONDA_ENV/'lib'),str(CONDA_ENV/'lib64'),str(CONDA_ENV/'targets'/'x86_64-linux'/'lib')]
build_env['LD_LIBRARY_PATH']=os.pathsep.join(lib_dirs+[build_env.get('LD_LIBRARY_PATH','')])
subprocess.run([
    str(CONDA_PY),'-m','pip','install','--no-build-isolation',
    'git+https://github.com/facebookresearch/pytorch3d.git@stable',
],env=build_env,check=True)
CONDA_RUNTIME_ENV=build_env.copy()
print('Conda Python 3.11 + PyTorch3D setup finished.')

## 3) التحقق من CUDA وFPS الأصلي داخل Conda

هذا الفحص يستخدم مفسر Python 3.11 المعزول، لا نواة Python 3.13 في Colab. لا نتابع قبل نجاحه.


In [ ]:
import subprocess
probe=r"""
import sys, torch, pytorch3d
from pytorch3d.ops import sample_farthest_points
print('Python:',sys.version.split()[0])
print('PyTorch:',torch.__version__,'| CUDA:',torch.version.cuda)
print('PyTorch3D:',pytorch3d.__file__)
assert torch.cuda.is_available(),'CUDA unavailable in Conda Python'
print('GPU:',torch.cuda.get_device_name(0))
x=torch.rand(1,32,3,device='cuda')
y,idx=sample_farthest_points(x,K=8)
assert y.shape==(1,8,3) and idx.shape==(1,8)
print('Native CUDA FPS passed:',tuple(y.shape))
"""
subprocess.run([str(CONDA_PY),'-c',probe],env=CONDA_RUNTIME_ENV,check=True)

## 4) تحميل ملفات الحالة من Google Drive
لا يقرأ هذا القسم ملفات العينة من GitHub. ضع ZIP في Google Drive كما هو موضح أعلى الدفتر؛ إذا لم يجده، سيطلب منك رفعه من اللابتوب. تبقى نسخة العمل داخل جلسة Colab المؤقتة.

In [ ]:
from google.colab import drive
import zipfile, shutil
drive.mount('/content/drive')
SAMPLE_ZIP = Path('/content/drive/MyDrive/DentalAI/case_46_candidate_01.zip')
if not SAMPLE_ZIP.is_file():
    from google.colab import files
    print('لم أجد ZIP في Drive؛ اختر case_46_candidate_01.zip من اللابتوب لرفعه إلى Colab.')
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith('.zip')]
    if len(candidates) != 1:
        raise FileNotFoundError('ارفع ZIP واحدًا بالضبط يحتوي ملفات العينة الخمسة.')
    SAMPLE_ZIP = Path('/content') / candidates[0]
required = ['crown.ply', 'pna_crop.ply', 'boundary_labels.npy', 'crown_attributes.h5', 'psr_my.npz']
extract_dir = Path('/content/case46_upload')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with zipfile.ZipFile(SAMPLE_ZIP) as zf:
    members = zf.namelist()
    if any(Path(m).is_absolute() or '..' in Path(m).parts for m in members):
        raise ValueError('ZIP يحتوي مسارًا غير آمن؛ أوقفنا فك الضغط.')
    zf.extractall(extract_dir)
found = {}
for name in required:
    matches = list(extract_dir.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f'توقعنا ملفًا واحدًا باسم {name} داخل ZIP، وجدنا {len(matches)}')
    found[name] = matches[0]
DATA_ROOT = Path('/content/madcrowner_onecase')
CASE_DIR = DATA_ROOT / '46' / 'train' / 'case_46_candidate_01'
CASE_DIR.mkdir(parents=True, exist_ok=True)
for name, src in found.items():
    shutil.copy2(src, CASE_DIR / name)
print('Loaded exactly five sample files into:', CASE_DIR)
print('Files:', ', '.join(sorted(p.name for p in CASE_DIR.iterdir())))

## 5) اختبار IOS_Dataset مع PyTorch3D FPS الأصلي
نستخدم عدد نقاط أقل في هذا smoke test لتقليل احتمال نفاد ذاكرة T4. هذا يختبر قراءة البيانات وامتداد FPS الأصلي، ولا يغيّر إعداد التدريب النهائي.
يُشغّل الاختبار بوضوح عبر `CONDA_PY` (Python 3.11)، مع FPS الأصلي CUDA.


In [ ]:
import subprocess
smoke_script=REPO_DIR/'scripts'/'colab_gpu_smoke_test.py'
subprocess.run([
    str(CONDA_PY),str(smoke_script),'--phase','dataset','--data-root',str(DATA_ROOT),
],env=CONDA_RUNTIME_ENV,check=True)

## 6) عشر خطوات تدريب تجريبية على CrownDeformer
يستخدم الاختبار resolution أصغر للـPSR (`32³`) وعدد نقاط محدود حتى يناسب T4. يُعاد تحجيم target PSR إلى هذا المقاس. هذه ليست إعدادات تدريب نهائي ولا تقيس جودة التاج.
يُشغّل forward/backward/optimizer عشر مرات من سكربت المشروع داخل بيئة Conda. تبدأ الأوزان عشوائيًا، والـcheckpoint للتأكد التقني فقط.


In [ ]:
import subprocess
checkpoint_dir=Path('/content/drive/MyDrive/DentalAI/checkpoints')
checkpoint_path=checkpoint_dir/'case46_colab_smoke_10steps.pt'
smoke_script=REPO_DIR/'scripts'/'colab_gpu_smoke_test.py'
subprocess.run([
    str(CONDA_PY),str(smoke_script),'--phase','train',
    '--data-root',str(DATA_ROOT),'--checkpoint',str(checkpoint_path),
],env=CONDA_RUNTIME_ENV,check=True)

## 7) التحقق من حفظ checkpoint التجريبي على Google Drive
الملف الناتج يحفظ حالة النموذج والـoptimizer بعد 10 خطوات. احتفظ به بوصفه **checkpoint لاختبار أن المسار يعمل فقط**؛ لا تستخدمه للاستدلال السريري.

In [ ]:
assert checkpoint_path.is_file(),f'Checkpoint not found: {checkpoint_path}'
print('Checkpoint is present on Google Drive:',checkpoint_path)
print('Size (MiB):',round(checkpoint_path.stat().st_size/1024**2,2))